# Amazon ML Challenge — Matcher V1

## Objective

Build the first baseline entity-matching model using labeled S1–S2/S3
candidate pairs.

### Input

A candidate-pair table containing:

- `source1_entity_id`
- `candidate_entity_id`

with the candidate entity's source record available for feature generation.

### Output

A match probability for each candidate pair:

- `1` → likely same business
- `0` → likely different businesses

The matcher will not enforce one-to-one matching. Multiple candidates may
be predicted as matches for the same S1 entity, and an S1 entity may have
zero predicted matches.

### Day 1 Goal

Establish a reproducible baseline matcher and evaluate it using
entity-level macro F0.5 on the held-out validation S1 entities.

## 1. Matcher Interface

The matcher consumes the final candidate set produced by the candidate
generation stage.

Required candidate-pair columns:

- `source1_entity_id`
- `candidate_entity_id`

The candidate entity ID must refer to an existing S2 or S3 record.

The matcher is responsible for:

1. Constructing pair-level features
2. Assigning training labels from ground truth
3. Training a baseline binary classifier
4. Producing match probabilities
5. Selecting a decision threshold
6. Evaluating predictions using entity-level macro F0.5

In [9]:
from pathlib import Path
import pandas as pd
import numpy as np

from sklearn.metrics import (
    precision_score,
    recall_score,
    fbeta_score,
    classification_report
)

RANDOM_STATE = 42
CHUNK_SIZE = 100_000

In [2]:
PROJECT_ROOT = Path("..")

DATASET_DIR = PROJECT_ROOT / "dataset"
TRAIN_DIR = DATASET_DIR / "train"

TRAIN_S1 = TRAIN_DIR / "train_source1.tsv"
TRAIN_S2 = TRAIN_DIR / "train_source2.tsv"
TRAIN_S3 = TRAIN_DIR / "train_source3.tsv"
TRAIN_GT = TRAIN_DIR / "train_ground_truth.tsv"

print("S1:", TRAIN_S1.exists())
print("S2:", TRAIN_S2.exists())
print("S3:", TRAIN_S3.exists())
print("GT:", TRAIN_GT.exists())

S1: True
S2: True
S3: True
GT: True


In [3]:
from sklearn.model_selection import train_test_split

gt = pd.read_csv(
    TRAIN_GT,
    sep="\t",
    keep_default_na=False
)

all_s1_ids = gt["source1_entity_id"].values

train_s1_ids, valid_s1_ids = train_test_split(
    all_s1_ids,
    test_size=0.20,
    random_state=RANDOM_STATE
)

train_s1_set = set(train_s1_ids)
valid_s1_set = set(valid_s1_ids)

print("Training S1:", len(train_s1_set))
print("Validation S1:", len(valid_s1_set))

assert not (train_s1_set & valid_s1_set)

print("✓ Entity-level split reproduced successfully.")

Training S1: 1765456
Validation S1: 441365
✓ Entity-level split reproduced successfully.


In [4]:
DEV_S1_SIZE = 5_000

rng = np.random.default_rng(RANDOM_STATE)

dev_s1_ids = rng.choice(
    np.array(list(train_s1_set)),
    size=DEV_S1_SIZE,
    replace=False
)

dev_s1_ids = set(dev_s1_ids)

print("Development S1 entities:", len(dev_s1_ids))

Development S1 entities: 5000


In [5]:
dev_gt = gt[
    gt["source1_entity_id"].isin(dev_s1_ids)
].copy()

dev_gt["matched_ids"] = dev_gt["matched_entity_ids"].apply(
    lambda x: [] if x.strip() == "" else x.split(",")
)

print("Development GT rows:", len(dev_gt))

print(
    "S1 entities with at least one true match:",
    (dev_gt["matched_entity_ids"].str.strip() != "").sum()
)

print(
    "S1 entities with no true match:",
    (dev_gt["matched_entity_ids"].str.strip() == "").sum()
)

Development GT rows: 5000
S1 entities with at least one true match: 4729
S1 entities with no true match: 271


In [6]:
positive_pairs = []

for _, row in dev_gt.iterrows():
    s1_id = row["source1_entity_id"]

    for candidate_id in row["matched_ids"]:
        positive_pairs.append({
            "source1_entity_id": s1_id,
            "candidate_entity_id": candidate_id,
            "label": 1
        })

positive_pairs = pd.DataFrame(positive_pairs)

print("Positive pairs:", len(positive_pairs))
display(positive_pairs.head(10))

Positive pairs: 17449


,source1_entity_id,candidate_entity_id,label
0,S1-226964528,S2-188295837,1
1,S1-226964528,S2-74003616,1
2,S1-226964528,S2-625675086,1
3,S1-226964528,S3-560636619,1
4,S1-226964528,S3-709974281,1
5,S1-226964528,S3-180885195,1
6,S1-720511261,S2-230935368,1
7,S1-720511261,S2-888152272,1
8,S1-720511261,S3-810965798,1
9,S1-720511261,S3-264208857,1


In [7]:
positive_count = len(positive_pairs)

print(f"Positive pairs: {positive_count:,}")

Positive pairs: 17,449


## 2. Temporary Development Negatives

The final candidate set will eventually come from the blocking pipeline.

Because the blocking pipeline is not available yet, we create a temporary
development-only negative set for Matcher V1 development.

Negative candidates are sampled from the same country as the S1 entity.
This avoids making the baseline task artificially easy through country
mismatch alone.

These temporary candidates will NOT be used as the final
`candidate_pairs.tsv`.

In [10]:
def retrieve_records_by_ids(
    path,
    entity_ids,
    chunksize=CHUNK_SIZE
):
    entity_ids = set(entity_ids)
    results = []

    for chunk in pd.read_csv(
        path,
        sep="\t",
        chunksize=chunksize
    ):
        matched = chunk[
            chunk["entity_id"].isin(entity_ids)
        ]

        if not matched.empty:
            results.append(matched)

        if sum(len(x) for x in results) >= len(entity_ids):
            break

    if not results:
        return pd.DataFrame()

    return pd.concat(
        results,
        ignore_index=True
    )

In [11]:
dev_s1 = retrieve_records_by_ids(
    TRAIN_S1,
    dev_s1_ids
)

print("Retrieved S1 records:", len(dev_s1))
display(dev_s1.head())

Retrieved S1 records: 5000


,entity_id,business_name,business_address,country
0,S1-500552277,U Q & Z Kimco,"105 Susan Court, Winchester, KY",US
1,S1-581797355,Virtus Lifespaces Private Limited,"Sy No 10/28, Ambalipura, Bangalore South, Bang...",India
2,S1-430604402,Countryman Oxford Inc,"12360 502, Unit Gw2h, Venus, TX",US
3,S1-832785535,R X & A Avalanche,"1622 70th Street, Village Of Superior, WI",US
4,S1-697294833,Shiva Media Private Limited,"818B, The Camellias, Golf Course Road, Sector ...",India


## 3. Temporary Candidate Pool

The real candidate pool will eventually be produced by the teammate's
blocking pipeline.

For Matcher V1 development, we construct a temporary pool of S2/S3 records
from the same countries as the development S1 entities.

This pool is used only to generate development negatives and will not become
the final `candidate_pairs.tsv`.

In [12]:
DEV_POOL_SIZE = 100_000

dev_countries = set(dev_s1["country"].dropna().unique())

print("Development countries:", dev_countries)

Development countries: {'US', 'India'}


In [13]:
def sample_records_by_country(
    path,
    countries,
    target_size=100_000,
    chunksize=CHUNK_SIZE,
    random_state=RANDOM_STATE
):
    rng = np.random.default_rng(random_state)
    samples = []

    for chunk in pd.read_csv(
        path,
        sep="\t",
        chunksize=chunksize
    ):
        chunk = chunk[
            chunk["country"].isin(countries)
        ]

        if not chunk.empty:
            samples.append(chunk)

    pool = pd.concat(
        samples,
        ignore_index=True
    )

    if len(pool) > target_size:
        indices = rng.choice(
            len(pool),
            size=target_size,
            replace=False
        )
        pool = pool.iloc[indices].reset_index(drop=True)

    return pool

In [14]:
s2_pool = sample_records_by_country(
    TRAIN_S2,
    dev_countries,
    target_size=DEV_POOL_SIZE,
    random_state=RANDOM_STATE
)

s3_pool = sample_records_by_country(
    TRAIN_S3,
    dev_countries,
    target_size=DEV_POOL_SIZE,
    random_state=RANDOM_STATE + 1
)

candidate_pool = pd.concat(
    [s2_pool, s3_pool],
    ignore_index=True
)

print("S2 pool:", len(s2_pool))
print("S3 pool:", len(s3_pool))
print("Combined candidate pool:", len(candidate_pool))

display(candidate_pool.head())

S2 pool: 100000
S3 pool: 100000
Combined candidate pool: 200000


,entity_id,business_name,business_address,country
0,S2-796341195,"GONZALEZ, BONITA DISCIPLINED PARTNERS","71-D FLETCHER STREET, GOSHEN, NY",US
1,S2-482237115,Blanchard Higth Center,"2520 PLAZA DR, TEMPE, AZ",US
2,S2-508345922,(india) Tfs Másale,"1 , SHOP NO. 3, VASUDEV INDUSTRIES ESTATE, KAL...",India
3,S2-316521483,"Empire Dymic Sachem, Holdings","11103 CARTER CANYON ROAD, MOUNT LEMMON, AZ",US
4,S2-234103134,குஜராத் யுனைடெட் பிசினஸ் பிரைவேட் லிமிடெட்,"D NO: 60/1/2 VELLIKALMEDU, ILAIYANELLORE POST,...",India


## 4. Temporary Negative Pair Generation

For Matcher V1 development, create non-matching S1–S2/S3 pairs from the
same country.

True ground-truth matches are explicitly excluded.

This is a temporary development dataset only. The final candidate set will
later be supplied by the blocking pipeline.

In [15]:
NEGATIVE_RATIO = 1.0

# Map each S1 entity to its true matched IDs
true_matches = {
    row["source1_entity_id"]: set(row["matched_ids"])
    for _, row in dev_gt.iterrows()
}

# Candidate pool grouped by country
candidate_by_country = {
    country: group.reset_index(drop=True)
    for country, group in candidate_pool.groupby("country")
}

rng = np.random.default_rng(RANDOM_STATE)

negative_pairs = []

for _, s1_row in dev_s1.iterrows():
    s1_id = s1_row["entity_id"]
    country = s1_row["country"]

    pool = candidate_by_country.get(country)

    if pool is None or len(pool) == 0:
        continue

    forbidden = true_matches.get(s1_id, set())

    # Sample a few candidates and keep only non-matches
    sample_size = 10

    indices = rng.integers(
        0,
        len(pool),
        size=sample_size
    )

    selected = pool.iloc[indices]

    for _, candidate in selected.iterrows():

        candidate_id = candidate["entity_id"]

        if candidate_id in forbidden:
            continue

        negative_pairs.append({
            "source1_entity_id": s1_id,
            "candidate_entity_id": candidate_id,
            "label": 0
        })

        if len(negative_pairs) >= int(
            len(positive_pairs) * NEGATIVE_RATIO
        ):
            break

    if len(negative_pairs) >= int(
        len(positive_pairs) * NEGATIVE_RATIO
    ):
        break

negative_pairs = pd.DataFrame(negative_pairs)

print(f"Positive pairs: {len(positive_pairs):,}")
print(f"Negative pairs: {len(negative_pairs):,}")

Positive pairs: 17,449
Negative pairs: 17,449


In [16]:
negative_check = []

for _, row in negative_pairs.iterrows():
    true_ids = true_matches.get(
        row["source1_entity_id"],
        set()
    )

    negative_check.append(
        row["candidate_entity_id"] in true_ids
    )

print(
    "Incorrect negative pairs:",
    sum(negative_check)
)

assert sum(negative_check) == 0

print("✓ No ground-truth positive pair was included as a negative.")

Incorrect negative pairs: 0
✓ No ground-truth positive pair was included as a negative.


In [17]:
dev_pairs = pd.concat(
    [
        positive_pairs,
        negative_pairs
    ],
    ignore_index=True
)

# Shuffle
dev_pairs = dev_pairs.sample(
    frac=1,
    random_state=RANDOM_STATE
).reset_index(drop=True)

print("Development pair dataset:", len(dev_pairs))

display(dev_pairs.head(10))

print("\nLabel distribution:")
display(
    dev_pairs["label"]
    .value_counts()
    .sort_index()
)

Development pair dataset: 34898


,source1_entity_id,candidate_entity_id,label
0,S1-784291484,S3-610604164,0
1,S1-592818843,S3-885788641,0
2,S1-131201345,S3-232351166,0
3,S1-912953970,S3-656134924,0
4,S1-596144789,S2-398739678,1
5,S1-181549964,S2-154173552,1
6,S1-698253562,S3-165956992,0
7,S1-761602930,S2-246716777,0
8,S1-860778681,S2-499263497,1
9,S1-974099083,S2-705499639,1



Label distribution:


label
0    17449
1    17449
Name: count, dtype: int64

## 5. Retrieve Pair Records

Retrieve the S1, S2, and S3 records required by the development pair
dataset.

Only records appearing in the development pairs are retrieved; the complete
source datasets are not loaded into memory.

In [18]:
# Collect all S1 IDs used in the development pairs
pair_s1_ids = set(dev_pairs["source1_entity_id"])

# Split candidate IDs by source
pair_s2_ids = set(
    dev_pairs.loc[
        dev_pairs["candidate_entity_id"].str.startswith("S2-"),
        "candidate_entity_id"
    ]
)

pair_s3_ids = set(
    dev_pairs.loc[
        dev_pairs["candidate_entity_id"].str.startswith("S3-"),
        "candidate_entity_id"
    ]
)

print("S1 IDs needed:", len(pair_s1_ids))
print("S2 IDs needed:", len(pair_s2_ids))
print("S3 IDs needed:", len(pair_s3_ids))

S1 IDs needed: 4826
S2 IDs needed: 16752
S3 IDs needed: 17400


In [19]:
s1_pair_records = retrieve_records_by_ids(
    TRAIN_S1,
    pair_s1_ids
)

s2_pair_records = retrieve_records_by_ids(
    TRAIN_S2,
    pair_s2_ids
)

s3_pair_records = retrieve_records_by_ids(
    TRAIN_S3,
    pair_s3_ids
)

candidate_pair_records = pd.concat(
    [
        s2_pair_records,
        s3_pair_records
    ],
    ignore_index=True
)

print("S1 records retrieved:", len(s1_pair_records))
print("S2 records retrieved:", len(s2_pair_records))
print("S3 records retrieved:", len(s3_pair_records))
print("Candidate records retrieved:", len(candidate_pair_records))

S1 records retrieved: 4826
S2 records retrieved: 16752
S3 records retrieved: 17400
Candidate records retrieved: 34152


In [20]:
assert len(s1_pair_records) == len(pair_s1_ids), (
    "Some required S1 records were not retrieved."
)

assert len(s2_pair_records) == len(pair_s2_ids), (
    "Some required S2 records were not retrieved."
)

assert len(s3_pair_records) == len(pair_s3_ids), (
    "Some required S3 records were not retrieved."
)

print("✓ All records required for Matcher V1 were retrieved.")

✓ All records required for Matcher V1 were retrieved.


In [21]:
display(s1_pair_records.head())
display(candidate_pair_records.head())

,entity_id,business_name,business_address,country
0,S1-500552277,U Q & Z Kimco,"105 Susan Court, Winchester, KY",US
1,S1-581797355,Virtus Lifespaces Private Limited,"Sy No 10/28, Ambalipura, Bangalore South, Bang...",India
2,S1-430604402,Countryman Oxford Inc,"12360 502, Unit Gw2h, Venus, TX",US
3,S1-832785535,R X & A Avalanche,"1622 70th Street, Village Of Superior, WI",US
4,S1-697294833,Shiva Media Private Limited,"818B, The Camellias, Golf Course Road, Sector ...",India


,entity_id,business_name,business_address,country
0,S2-49942811,Delta Tetlecommunication Inc,"914 PIERPONT AVE, CLEVELAND, OH",US
1,S2-739448663,Shri @maalogistics,"Karnataka, NO.2307, (OLD NO.44/6), GROUND FLOO...",India
2,S2-310050325,"Dorris, Wright and Johnson of Covington Ltd","1 ROEBLING WAY, COVINGTON, KY",US
3,S2-569534268,Central Hospitality,"666 VI-LILLY CIR, GROVE CITY, OH",US
4,S2-328900052,"Pioneer-Financial, LLC","4249 HCIKORY RD, CHESTERFIELD COUNTY, VA",US


## 6. Build Pair-Level Dataset

Join each development pair with its corresponding S1 and S2/S3 records.

The resulting table will contain the raw fields needed to calculate
pair-level similarity features.

In [22]:
# Prepare S1 records
s1_features = s1_pair_records.rename(
    columns={
        "entity_id": "source1_entity_id",
        "business_name": "s1_business_name",
        "business_address": "s1_business_address",
        "country": "s1_country"
    }
)

# Prepare candidate records
candidate_features = candidate_pair_records.rename(
    columns={
        "entity_id": "candidate_entity_id",
        "business_name": "candidate_business_name",
        "business_address": "candidate_business_address",
        "country": "candidate_country"
    }
)

# Merge S1 information
pair_dataset = dev_pairs.merge(
    s1_features,
    on="source1_entity_id",
    how="left",
    validate="many_to_one"
)

# Merge candidate information
pair_dataset = pair_dataset.merge(
    candidate_features,
    on="candidate_entity_id",
    how="left",
    validate="many_to_one"
)

print("Pair dataset shape:", pair_dataset.shape)

display(pair_dataset.head())

Pair dataset shape: (34898, 9)


,source1_entity_id,candidate_entity_id,label,s1_business_name,s1_business_address,s1_country,candidate_business_name,candidate_business_address,candidate_country
0,S1-784291484,S3-610604164,0,Total Learning Works LLC,"110 College Street, Cedar Falls, IA",US,matthewsmicroelectronics.com,"Massachusetts, New Bedford, Unit 1 REAR, 296 S...",US
1,S1-592818843,S3-885788641,0,# 5 SG Arbor,"5550 Allison Mill Road, Talladega, AL",US,Llc Wyman Microelectronics,"1359 Carling Dr, # Apartment 338, Saint Paul, ...",US
2,S1-131201345,S3-232351166,0,Jones Express Specialty Inc.,"1S 67 Summit Avenue, Oakbrook Terrace, IL",US,Capital Carities VI LLC,"12810 Flagstone Dr, NULL, Sun City West, Arizona",US
3,S1-912953970,S3-656134924,0,"Robenia Vorce, CPA, PC","9243 Coffee Hill Road, West Sparta, NY",US,Moex Moex LLC,"167 Wooster Street, New Haven, Connecticut",US
4,S1-596144789,S2-398739678,1,Thompson Group,"502 1040, American Fork, UT",US,Thompson-Group,"502 1040, AMERICAN FORK, UT",US


In [23]:
assert len(pair_dataset) == len(dev_pairs)

required_columns = [
    "source1_entity_id",
    "candidate_entity_id",
    "s1_business_name",
    "s1_business_address",
    "s1_country",
    "candidate_business_name",
    "candidate_business_address",
    "candidate_country",
    "label"
]

missing_columns = [
    col for col in required_columns
    if col not in pair_dataset.columns
]

assert not missing_columns, f"Missing columns: {missing_columns}"

print("✓ Pair dataset successfully constructed.")

✓ Pair dataset successfully constructed.


In [24]:
print("POSITIVE PAIR")
display(
    pair_dataset[pair_dataset["label"] == 1].head(5)
)

print("\nNEGATIVE PAIR")
display(
    pair_dataset[pair_dataset["label"] == 0].head(5)
)

POSITIVE PAIR


,source1_entity_id,candidate_entity_id,label,s1_business_name,s1_business_address,s1_country,candidate_business_name,candidate_business_address,candidate_country
4,S1-596144789,S2-398739678,1,Thompson Group,"502 1040, American Fork, UT",US,Thompson-Group,"502 1040, AMERICAN FORK, UT",US
5,S1-181549964,S2-154173552,1,Carlene Dial Rocky Cafe LLC,"1112 28th Avenue, Albany, OR",US,carlenedialrocky.com,"OR, ALBANY, 1112 28ND AVE",US
8,S1-860778681,S2-499263497,1,Cook Dividend Inc,"Hood River, 4250 Willow Flat Road, OR",US,Cook Dividend,"250 WILLOW FLAT RD, HOOD RIVER, OR",US
9,S1-974099083,S2-705499639,1,Renet Assets,"618 Granite Street, Unit 112, Braintree, MA",US,RENET ASSETS,"618 GRANITE ST, BRRAINTREE, MA",US
10,S1-63390156,S2-115795636,1,Jain Jain Logistics Limited,"House No-877, 3Rd Floor Dr. Mukherjee Nagar, O...",India,जैन जैन लॉजिस्टिक्स लिमिटेड,"HOUSE NO-877, 3RD FLOOR DR. MUKHERJEE NAGAR, O...",India



NEGATIVE PAIR


,source1_entity_id,candidate_entity_id,label,s1_business_name,s1_business_address,s1_country,candidate_business_name,candidate_business_address,candidate_country
0,S1-784291484,S3-610604164,0,Total Learning Works LLC,"110 College Street, Cedar Falls, IA",US,matthewsmicroelectronics.com,"Massachusetts, New Bedford, Unit 1 REAR, 296 S...",US
1,S1-592818843,S3-885788641,0,# 5 SG Arbor,"5550 Allison Mill Road, Talladega, AL",US,Llc Wyman Microelectronics,"1359 Carling Dr, # Apartment 338, Saint Paul, ...",US
2,S1-131201345,S3-232351166,0,Jones Express Specialty Inc.,"1S 67 Summit Avenue, Oakbrook Terrace, IL",US,Capital Carities VI LLC,"12810 Flagstone Dr, NULL, Sun City West, Arizona",US
3,S1-912953970,S3-656134924,0,"Robenia Vorce, CPA, PC","9243 Coffee Hill Road, West Sparta, NY",US,Moex Moex LLC,"167 Wooster Street, New Haven, Connecticut",US
6,S1-698253562,S3-165956992,0,Dea River LLC,"534 Yosemite Avenue, City Of Hartford, WI",US,Dental Gulf Gulf Physicians LLC,"56 Victoria Lane, Marlborough, Massachusetts",US


## 7. Pair-Level Feature Engineering

Create interpretable similarity features for each S1–S2/S3 pair.

### Name features
- Normalized exact equality
- Character similarity
- Token Jaccard similarity

### Address features
- Normalized exact equality
- Character similarity
- Token Jaccard similarity

### Additional features
- Country equality
- Missing-name indicators
- Missing-address indicators

The features are computed independently for each candidate pair.

In [25]:
import rapidfuzz

print("RapidFuzz version:", rapidfuzz.__version__)

RapidFuzz version: 3.14.6


In [26]:
import re
import unicodedata

from rapidfuzz.fuzz import ratio


def normalize_text(value):
    """
    Lightweight normalization for Matcher V1.
    
    Keeps the raw data untouched and creates a comparison representation.
    """
    if pd.isna(value):
        return ""

    value = str(value).lower().strip()

    # Unicode normalization
    value = unicodedata.normalize("NFKC", value)

    # Replace punctuation with spaces
    value = re.sub(r"[^\w\s]", " ", value, flags=re.UNICODE)

    # Collapse whitespace
    value = re.sub(r"\s+", " ", value).strip()

    return value


def token_jaccard(a, b):
    """
    Jaccard similarity between token sets.
    """
    a_tokens = set(normalize_text(a).split())
    b_tokens = set(normalize_text(b).split())

    if not a_tokens and not b_tokens:
        return 1.0

    if not a_tokens or not b_tokens:
        return 0.0

    return len(a_tokens & b_tokens) / len(a_tokens | b_tokens)


def character_similarity(a, b):
    """
    Character-level similarity in [0, 1].
    """
    a_norm = normalize_text(a)
    b_norm = normalize_text(b)

    if not a_norm and not b_norm:
        return 1.0

    if not a_norm or not b_norm:
        return 0.0

    return ratio(a_norm, b_norm) / 100.0

In [27]:
def build_pair_features(df):
    features = pd.DataFrame(index=df.index)

    # -------------------------
    # Name normalization
    # -------------------------
    s1_name = df["s1_business_name"].map(normalize_text)
    cand_name = df["candidate_business_name"].map(normalize_text)

    features["name_exact"] = (
        (s1_name == cand_name) &
        (s1_name != "")
    ).astype(int)

    features["name_char_similarity"] = [
        character_similarity(a, b)
        for a, b in zip(
            df["s1_business_name"],
            df["candidate_business_name"]
        )
    ]

    features["name_token_jaccard"] = [
        token_jaccard(a, b)
        for a, b in zip(
            df["s1_business_name"],
            df["candidate_business_name"]
        )
    ]

    # -------------------------
    # Address
    # -------------------------
    s1_address = df["s1_business_address"].map(normalize_text)
    cand_address = df["candidate_business_address"].map(normalize_text)

    features["address_exact"] = (
        (s1_address == cand_address) &
        (s1_address != "")
    ).astype(int)

    features["address_char_similarity"] = [
        character_similarity(a, b)
        for a, b in zip(
            df["s1_business_address"],
            df["candidate_business_address"]
        )
    ]

    features["address_token_jaccard"] = [
        token_jaccard(a, b)
        for a, b in zip(
            df["s1_business_address"],
            df["candidate_business_address"]
        )
    ]

    # -------------------------
    # Country
    # -------------------------
    features["country_match"] = (
        df["s1_country"].fillna("").str.lower()
        ==
        df["candidate_country"].fillna("").str.lower()
    ).astype(int)

    # -------------------------
    # Missingness
    # -------------------------
    features["s1_name_missing"] = (
        df["s1_business_name"].isna() |
        (df["s1_business_name"].astype(str).str.strip() == "")
    ).astype(int)

    features["candidate_name_missing"] = (
        df["candidate_business_name"].isna() |
        (df["candidate_business_name"].astype(str).str.strip() == "")
    ).astype(int)

    features["s1_address_missing"] = (
        df["s1_business_address"].isna() |
        (df["s1_business_address"].astype(str).str.strip() == "")
    ).astype(int)

    features["candidate_address_missing"] = (
        df["candidate_business_address"].isna() |
        (df["candidate_business_address"].astype(str).str.strip() == "")
    ).astype(int)

    return features

In [28]:
X_dev = build_pair_features(pair_dataset)
y_dev = pair_dataset["label"].astype(int)

print("Feature matrix shape:", X_dev.shape)
print("\nFeatures:")
print(list(X_dev.columns))

display(X_dev.head())

Feature matrix shape: (34898, 11)

Features:
['name_exact', 'name_char_similarity', 'name_token_jaccard', 'address_exact', 'address_char_similarity', 'address_token_jaccard', 'country_match', 's1_name_missing', 'candidate_name_missing', 's1_address_missing', 'candidate_address_missing']


,name_exact,name_char_similarity,name_token_jaccard,address_exact,address_char_similarity,address_token_jaccard,country_match,s1_name_missing,candidate_name_missing,s1_address_missing,candidate_address_missing
0,0,0.384615,0.0,0,0.321839,0.0,1,0,0,0,0
1,0,0.277778,0.0,0,0.376471,0.0,1,0,0,0,0
2,0,0.320000,0.0,0,0.357143,0.0,1,0,0,0,0
3,0,0.424242,0.0,0,0.315789,0.0,1,0,0,0,0
4,1,1.000000,1.0,1,1.000000,1.0,1,0,0,0,0


## 8. Entity-Level Matcher Development Split

Split the development S1 entities into training and validation groups.

The split is performed at the S1 entity level so that pairs belonging to the
same S1 entity cannot appear in both training and validation.

This prevents pair-level leakage during Matcher V1 evaluation.

In [29]:
from sklearn.model_selection import train_test_split

pair_s1_ids = pair_dataset["source1_entity_id"].unique()

matcher_train_s1, matcher_valid_s1 = train_test_split(
    pair_s1_ids,
    test_size=0.20,
    random_state=RANDOM_STATE
)

matcher_train_s1 = set(matcher_train_s1)
matcher_valid_s1 = set(matcher_valid_s1)

print("Matcher training S1:", len(matcher_train_s1))
print("Matcher validation S1:", len(matcher_valid_s1))

assert not (matcher_train_s1 & matcher_valid_s1)

print("✓ No S1 entity appears in both matcher partitions.")

Matcher training S1: 3860
Matcher validation S1: 966
✓ No S1 entity appears in both matcher partitions.


In [30]:
train_mask = pair_dataset["source1_entity_id"].isin(
    matcher_train_s1
)

valid_mask = pair_dataset["source1_entity_id"].isin(
    matcher_valid_s1
)

X_train = X_dev.loc[train_mask].copy()
y_train = y_dev.loc[train_mask].copy()

X_valid = X_dev.loc[valid_mask].copy()
y_valid = y_dev.loc[valid_mask].copy()

print("Training pairs:", len(X_train))
print("Validation pairs:", len(X_valid))

print("\nTraining labels:")
print(y_train.value_counts())

print("\nValidation labels:")
print(y_valid.value_counts())

Training pairs: 27807
Validation pairs: 7091

Training labels:
label
1    14008
0    13799
Name: count, dtype: int64

Validation labels:
label
0    3650
1    3441
Name: count, dtype: int64


## 9. Matcher V1 — Logistic Regression Baseline

Train a simple binary classifier on the pair-level similarity features.

The model predicts:

- 1 → S1 and candidate are likely the same business
- 0 → S1 and candidate are likely different businesses

The model outputs probabilities so that the decision threshold can be tuned
for the precision-heavy F0.5 evaluation metric.

In [31]:
from sklearn.linear_model import LogisticRegression

matcher_v1 = LogisticRegression(
    max_iter=1000,
    random_state=RANDOM_STATE
)

matcher_v1.fit(
    X_train,
    y_train
)

print("✓ Matcher V1 trained.")

✓ Matcher V1 trained.


In [32]:
valid_probabilities = matcher_v1.predict_proba(
    X_valid
)[:, 1]

print("Validation probabilities:")
print(valid_probabilities[:10])

Validation probabilities:
[0.99999763 0.99999998 1.         0.02269219 1.         0.01514183
 1.         0.99998902 0.00279347 0.99999996]


In [33]:
thresholds = np.arange(
    0.10,
    0.96,
    0.05
)

threshold_results = []

for threshold in thresholds:

    predictions = (
        valid_probabilities >= threshold
    ).astype(int)

    precision = precision_score(
        y_valid,
        predictions,
        zero_division=0
    )

    recall = recall_score(
        y_valid,
        predictions,
        zero_division=0
    )

    f05 = fbeta_score(
        y_valid,
        predictions,
        beta=0.5,
        zero_division=0
    )

    threshold_results.append({
        "threshold": threshold,
        "precision": precision,
        "recall": recall,
        "f0.5": f05
    })

threshold_results = pd.DataFrame(
    threshold_results
)

display(threshold_results)

,threshold,precision,recall,f0.5
0,0.10,0.970272,0.995931,0.975297
1,0.15,0.980791,0.994188,0.983442
2,0.20,0.986728,0.993897,0.988154
3,0.25,0.991589,0.993607,0.991992
4,0.30,0.993603,0.993025,0.993487
5,0.35,0.995044,0.991863,0.994406
6,0.40,0.996495,0.991572,0.995507
7,0.45,0.997365,0.990119,0.995908
8,0.50,0.997945,0.988085,0.995958
9,0.55,0.998237,0.987213,0.996012


## 10. Entity-Level Validation

Evaluate Matcher V1 at the S1-entity level.

For each S1 entity:

1. Convert candidate probabilities into predicted matched IDs.
2. Compare the predicted set against the ground-truth matched IDs.
3. Calculate F0.5 for each S1 entity.
4. Macro-average across validation S1 entities.

This better reflects the actual challenge evaluation structure.

In [34]:
def entity_level_macro_f05(
    pair_df,
    probabilities,
    threshold,
    gt_lookup
):
    temp = pair_df.copy()

    temp["probability"] = probabilities
    temp["predicted"] = (
        temp["probability"] >= threshold
    )

    entity_scores = []

    for s1_id, group in temp.groupby("source1_entity_id"):

        predicted_ids = set(
            group.loc[
                group["predicted"],
                "candidate_entity_id"
            ]
        )

        true_ids = gt_lookup.get(
            s1_id,
            set()
        )

        tp = len(predicted_ids & true_ids)
        fp = len(predicted_ids - true_ids)
        fn = len(true_ids - predicted_ids)

        if tp + fp == 0:
            precision = 0.0
        else:
            precision = tp / (tp + fp)

        if tp + fn == 0:
            recall = 1.0
        else:
            recall = tp / (tp + fn)

        if precision == 0 and recall == 0:
            f05 = 0.0
        else:
            f05 = (
                1.25 * precision * recall
                / (0.25 * precision + recall)
            )

        entity_scores.append({
            "source1_entity_id": s1_id,
            "precision": precision,
            "recall": recall,
            "f0.5": f05,
            "num_predicted": len(predicted_ids),
            "num_true": len(true_ids)
        })

    scores = pd.DataFrame(entity_scores)

    return scores["f0.5"].mean(), scores

In [36]:
valid_gt_lookup = {}

for _, row in dev_gt[
    dev_gt["source1_entity_id"].isin(matcher_valid_s1)
].iterrows():

    matched = row["matched_entity_ids"].strip()

    if matched == "":
        valid_gt_lookup[row["source1_entity_id"]] = set()
    else:
        valid_gt_lookup[row["source1_entity_id"]] = set(
            matched.split(",")
        )

print("Validation S1 entities in GT:", len(valid_gt_lookup))

Validation S1 entities in GT: 966


In [37]:
entity_f05_060, entity_scores_060 = entity_level_macro_f05(
    pair_dataset.loc[valid_mask],
    valid_probabilities,
    threshold=0.60,
    gt_lookup=valid_gt_lookup
)

print(
    f"Entity-level macro F0.5 @ 0.60: "
    f"{entity_f05_060:.6f}"
)

display(entity_scores_060.head())

Entity-level macro F0.5 @ 0.60: 0.973900


,source1_entity_id,precision,recall,f0.5,num_predicted,num_true
0,S1-100930621,1.0,1.000000,1.000000,6,6
1,S1-101155060,1.0,0.666667,0.909091,2,3
2,S1-102602479,1.0,1.000000,1.000000,3,3
3,S1-103010167,1.0,1.000000,1.000000,2,2
4,S1-104169191,1.0,1.000000,1.000000,1,1


## 11. Entity-Level Threshold Tuning

Evaluate Matcher V1 across multiple probability thresholds using
entity-level macro F0.5.

The threshold will be selected using the validation entities only.

In [38]:
entity_threshold_results = []

for threshold in np.arange(0.30, 0.91, 0.05):

    score, _ = entity_level_macro_f05(
        pair_dataset.loc[valid_mask],
        valid_probabilities,
        threshold=threshold,
        gt_lookup=valid_gt_lookup
    )

    entity_threshold_results.append({
        "threshold": threshold,
        "entity_macro_f0.5": score
    })

entity_threshold_results = pd.DataFrame(
    entity_threshold_results
)

display(entity_threshold_results)

,threshold,entity_macro_f0.5
0,0.30,0.972317
1,0.35,0.973012
2,0.40,0.973891
3,0.45,0.973965
4,0.50,0.974019
5,0.55,0.973966
6,0.60,0.973900
7,0.65,0.973724
8,0.70,0.973441
9,0.75,0.973140


In [39]:
best_entity_threshold = (
    entity_threshold_results
    .loc[
        entity_threshold_results["entity_macro_f0.5"].idxmax()
    ]
)

print("Best threshold:")
print(best_entity_threshold)

Best threshold:
threshold            0.500000
entity_macro_f0.5    0.974019
Name: 4, dtype: float64
